In [14]:
import pandas as pd
import numpy as np

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(144415, 87) (36104, 87) (144415,) (36104,)


In [15]:
# Check every column for non-numeric values that sneaked in
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(col, X_train[col].unique()[:5])

In [10]:
for col in X_test.columns:
    if X_test[col].dtype == 'object':
        print(col, X_test[col].unique()[:5])

Days for shipment (scheduled) ['StandardScaler()']
Benefit per order ['StandardScaler()']
Sales per customer ['StandardScaler()']
Category Id ['StandardScaler()']
Department Id ['StandardScaler()']
Order Item Discount ['StandardScaler()']
Order Item Discount Rate ['StandardScaler()']
Order Item Product Price ['StandardScaler()']
Order Item Profit Ratio ['StandardScaler()']
Order Item Quantity ['StandardScaler()']
Sales ['StandardScaler()']
Order Item Total ['StandardScaler()']
Order Profit Per Order ['StandardScaler()']
Product Category Id ['StandardScaler()']
Product Price ['StandardScaler()']
Product Status ['StandardScaler()']
order_month ['StandardScaler()']
order_dow ['StandardScaler()']
is_weekend ['StandardScaler()']
Type_CASH ['StandardScaler()']
Type_DEBIT ['StandardScaler()']
Type_PAYMENT ['StandardScaler()']
Type_TRANSFER ['StandardScaler()']
Customer Segment_Consumer ['StandardScaler()']
Customer Segment_Corporate ['StandardScaler()']
Customer Segment_Home Office ['Standard

In [16]:
print(y_train.dtype, y_test.dtype)
print(X_train.dtypes.value_counts())
print(X_test.dtypes.value_counts())

int64 int64
float64    87
Name: count, dtype: int64
float64    87
Name: count, dtype: int64


In [17]:
for col in X_test.columns[:5]:
    print(col, X_test[col].unique()[:3])

Days for shipment (scheduled) [ 0.77756685 -1.40522588 -0.6776283 ]
Benefit per order [-0.1027611  -0.11502573  0.90940377]
Sales per customer [-0.05714187  0.51683613  0.51600434]
Category Id [ 1.03430162 -0.50007481  0.90643692]
Department Id [ 0.95626437 -0.27071444 -1.49769325]


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [19]:
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, accuracy_score
import time

In [24]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000,random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42,n_jobs=-1),
    "Hist Grad": HistGradientBoostingClassifier(random_state=42),
    "XgBoost": XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1),
    "lightgbm": LGBMClassifier(random_state=42,verbose=-1,n_jobs=-1),
    "CatBoost": CatBoostClassifier(random_state=42,verbose=0)

}

In [25]:
results = []
for name, model in models.items():
    model.fit(X_train,y_train)
    y_pred =model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba),
    })

/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_s

In [26]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
results_df

,Model,Accuracy,F1,ROC-AUC,PR-AUC
1,Random Forest,0.757174,0.752100,0.849709,0.885462
5,CatBoost,0.743380,0.729434,0.840828,0.879966
3,XgBoost,0.734323,0.722068,0.820258,0.867628
4,lightgbm,0.717898,0.687279,0.809141,0.861563
2,Hist Grad,0.716957,0.686081,0.809202,0.861229
0,Logistic Regression,0.707512,0.668654,0.767267,0.834677


In [27]:
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, accuracy_score

rf_model = models['Random Forest']  # reuse the already-trained model from your loop

# Predictions on TRAIN
y_train_pred = rf_model.predict(X_train)
y_train_proba = rf_model.predict_proba(X_train)[:, 1]

# Predictions on TEST
y_test_pred = rf_model.predict(X_test)
y_test_proba = rf_model.predict_proba(X_test)[:, 1]

overfit_check = pd.DataFrame({
    'Metric': ['Accuracy', 'F1', 'ROC-AUC', 'PR-AUC'],
    'Train': [
        accuracy_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
        roc_auc_score(y_train, y_train_proba),
        average_precision_score(y_train, y_train_proba)
    ],
    'Test': [
        accuracy_score(y_test, y_test_pred),
        f1_score(y_test, y_test_pred),
        roc_auc_score(y_test, y_test_proba),
        average_precision_score(y_test, y_test_proba)
    ]
})

overfit_check['Gap'] = overfit_check['Train'] - overfit_check['Test']
overfit_check

,Metric,Train,Test,Gap
0,Accuracy,0.999993,0.757174,0.242819
1,F1,0.999994,0.752100,0.247894
2,ROC-AUC,1.000000,0.849709,0.150291
3,PR-AUC,1.000000,0.885462,0.114538


In [33]:
overfit_results = []

for name, model in models.items():
    y_train_pred = model.predict(X_train)
    y_train_proba = model.predict_proba(X_train)[:, 1]

    y_test_pred = model.predict(X_test)
    y_test_proba = model.predict_proba(X_test)[:, 1]

    train_f1 = f1_score(y_train, y_train_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    accuracy_train  = accuracy_score(y_train,y_train_pred)
    accuracy_test = accuracy_score(y_test,y_test_pred)

    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)

    overfit_results.append({
        'Model': name,
        'Accuracy_train': accuracy_train,
        'Accuracy_test': accuracy_test ,
        'Train F1': train_f1,
        'Test F1': test_f1,
        'F1 Gap': train_f1 - test_f1,
        'Train ROC-AUC': train_auc,
        'Test ROC-AUC': test_auc,
        'AUC Gap': train_auc - test_auc
    })

overfit_df = pd.DataFrame(overfit_results).sort_values('F1 Gap', ascending=False)
overfit_df

/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/parasgupta/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: Run

,Model,Accuracy_train,Accuracy_test,Train F1,Test F1,F1 Gap,Train ROC-AUC,Test ROC-AUC,AUC Gap
1,Random Forest,0.999993,0.757174,0.999994,0.752100,0.247894,1.000000,0.849709,0.150291
5,CatBoost,0.777385,0.743380,0.768554,0.729434,0.039120,0.892659,0.840828,0.051831
3,XgBoost,0.765835,0.734323,0.756885,0.722068,0.034818,0.869982,0.820258,0.049723
4,lightgbm,0.727805,0.717898,0.700066,0.687279,0.012787,0.842689,0.809141,0.033549
2,Hist Grad,0.726822,0.716957,0.698831,0.686081,0.012750,0.840805,0.809202,0.031603
0,Logistic Regression,0.710785,0.707512,0.673616,0.668654,0.004962,0.773000,0.767267,0.005734


In [37]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

xgb_param_dist = {
  'n_estimators': [100, 150, 200],
    'max_depth': [3, 4, 5],              # capped lower than before
    'learning_rate': [0.01, 0.03, 0.05],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [3, 5, 7, 10],   # raised — forces bigger leaf groups
    'gamma': [0.1, 0.2, 0.5, 1],         # raised — penalizes low-gain splits
    'reg_alpha': [0.1, 0.5, 1, 2],       # raised
    'reg_lambda': [2, 3, 5, 7],
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring='f1',
    cv=cv_strategy,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

print("Best params:", xgb_search.best_params_)
print("Best CV F1:", xgb_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=4, min_child_weight=10, n_estimators=200, reg_alpha=0.5, reg_lambda=7, subsample=0.7; total time=   3.0s
[CV] END colsample_bytree=0.6, gamma=1, learning_rate=0.01, max_depth=3, min_child_weight=5, n_estimators=200, reg_alpha=0.1, reg_lambda=3, subsample=0.6; total time=   3.0s
[CV] END colsample_bytree=0.6, gamma=1, learning_rate=0.01, max_depth=3, min_child_weight=5, n_estimators=200, reg_alpha=0.1, reg_lambda=3, subsample=0.6; total time=   3.0s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=4, min_child_weight=10, n_estimators=200, reg_alpha=0.5, reg_lambda=7, subsample=0.7; total time=   3.1s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.05, max_depth=4, min_child_weight=10, n_estimators=200, reg_alpha=0.5, reg_lambda=7, subsample=0.7; total time=   3.1s
[CV] END colsample_bytree=0.8, gamma=0.5, learning_rate=0.0

In [38]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, classification_report, confusion_matrix
)

best_xgb = xgb_search.best_estimator_

y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Test PR-AUC:", average_precision_score(y_test, y_proba))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7060436516729448
Test F1: 0.6845593698915143
Test ROC-AUC: 0.7683157734081816
Test PR-AUC: 0.8352230606861337

              precision    recall  f1-score   support

           0       0.63      0.86      0.72     16307
           1       0.83      0.58      0.68     19797

    accuracy                           0.71     36104
   macro avg       0.73      0.72      0.70     36104
weighted avg       0.74      0.71      0.70     36104


[[13975  2332]
 [ 8281 11516]]


In [39]:
y_train_pred = best_xgb.predict(X_train)
train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_pred)
print("Train F1:", train_f1, "Test F1:", test_f1, "Gap:", train_f1 - test_f1)

Train F1: 0.6882558691012568 Test F1: 0.6845593698915143 Gap: 0.0036964992097424654


In [ ]:
import joblib
import os

os.makedirs('../artifacts', exist_ok=True)


from xgboost import XGBClassifier

final_model = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
final_model.fit(X_train, y_train)

joblib.dump(final_model, '../artifacts/xgb_model.pkl')
print("Model saved to ../artifacts/xgb_model.pkl")

Model saved to ../artifacts/xgb_model.pkl


Exception ignored in: <function ResourceTracker.__del__ at 0x105f009a0>
Traceback (most recent call last):
  File "/Users/parasgupta/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/parasgupta/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/parasgupta/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1039009a0>
Traceback (most recent call last):
  File "/Users/parasgupta/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/parasgupta/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/parasgupta/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <fun